# 🧩 Notebook 2: UML → Python

## 🛠️ Setup

```bash
cd 07-object-oriented-design/uml-basics
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In this notebook we turn a UML sketch into real Python code, so you can see the mapping
from diagram → class → relationship.

We'll model the Library from notebook 1.

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from abc import ABC, abstractmethod


# ─────────────────────────────────────────────
# Author ──(1..*)── Book   (association, 1 author per book)
# ─────────────────────────────────────────────
@dataclass
class Author:
    name: str


# ─────────────────────────────────────────────
# Book (abstract base) △ PrintBook, Ebook, Audiobook   (inheritance)
# ─────────────────────────────────────────────
class Book(ABC):
    def __init__(self, title: str, author: Author):
        self.title = title
        self.author = author

    @abstractmethod
    def media_type(self) -> str: ...

    def __repr__(self):
        return f"{self.media_type()}({self.title!r} by {self.author.name})"


class PrintBook(Book):
    def media_type(self): return "PrintBook"

class Ebook(Book):
    def media_type(self): return "Ebook"

class Audiobook(Book):
    def media_type(self): return "Audiobook"


In [2]:
# ─────────────────────────────────────────────
# Library ──◆── Book  (composition: Library owns its Books)
# ─────────────────────────────────────────────
@dataclass
class Library:
    name: str
    books: list[Book] = field(default_factory=list)

    def add(self, book: Book) -> None:
        self.books.append(book)

    def by_author(self, author_name: str) -> list[Book]:
        return [b for b in self.books if b.author.name == author_name]


tolkien = Author("J.R.R. Tolkien")
rowling = Author("J.K. Rowling")

lib = Library("Central")
lib.add(PrintBook("The Hobbit", tolkien))
lib.add(Ebook("LOTR", tolkien))
lib.add(Audiobook("Harry Potter 1", rowling))

for b in lib.books:
    print(b)

print("Tolkien books:", lib.by_author("J.R.R. Tolkien"))


PrintBook('The Hobbit' by J.R.R. Tolkien)
Ebook('LOTR' by J.R.R. Tolkien)
Audiobook('Harry Potter 1' by J.K. Rowling)
Tolkien books: [PrintBook('The Hobbit' by J.R.R. Tolkien), Ebook('LOTR' by J.R.R. Tolkien)]


## Simulating a sequence diagram

The earlier sequence diagram becomes a chain of method calls. Watching the print output
is essentially the same thing as reading the sequence diagram top-to-bottom.

In [3]:
class PaymentSvc:
    def charge(self, user, amount):
        print(f"  PaymentSvc.charge({user}, ${amount}) → ok")
        return "ok"

class OrderSvc:
    def __init__(self, payments: PaymentSvc):
        self.payments = payments
        self._next_id = 41

    def create_order(self, user, amount):
        print(f" OrderSvc.create_order({user})")
        self.payments.charge(user, amount)
        self._next_id += 1
        return self._next_id

class WebApp:
    def __init__(self, orders: OrderSvc):
        self.orders = orders

    def checkout(self, user, amount):
        print(f"WebApp.checkout({user}) — user clicked 'Pay'")
        order_id = self.orders.create_order(user, amount)
        print(f"WebApp returns 200 OK, order #{order_id}")
        return order_id

app = WebApp(OrderSvc(PaymentSvc()))
app.checkout("alice", 42)


WebApp.checkout(alice) — user clicked 'Pay'
 OrderSvc.create_order(alice)
  PaymentSvc.charge(alice, $42) → ok
WebApp returns 200 OK, order #42


42

## Composition vs aggregation — in code

The diagram difference (◆ filled vs ◇ hollow) becomes a **lifetime** difference in code.

- **Composition** (◆): the whole *owns* the parts. When the whole disappears, so do the parts.
  The parts are usually **created inside** the whole.
- **Aggregation** (◇): the whole *references* parts that exist independently. They are usually
  **passed in** from outside.


In [4]:
# Composition: a House creates its own Rooms. Destroy the house, the rooms go with it.
class Room:
    def __init__(self, name): self.name = name
    def __repr__(self): return f"Room({self.name!r})"

class House:
    def __init__(self, room_names):
        # rooms are *created here* — no other object has a reference
        self._rooms = [Room(n) for n in room_names]
    @property
    def rooms(self): return list(self._rooms)

h = House(["kitchen", "bedroom"])
print("House owns:", h.rooms)


# Aggregation: a Team references Players that exist independently.
class Player:
    def __init__(self, name): self.name = name
    def __repr__(self): return f"Player({self.name!r})"

class Team:
    def __init__(self, name): self.name, self._players = name, []
    def sign(self, p: Player): self._players.append(p)  # passed in
    @property
    def players(self): return list(self._players)

alice, bob = Player("Alice"), Player("Bob")
red  = Team("Red");  red.sign(alice);  red.sign(bob)
blue = Team("Blue"); blue.sign(alice)   # Alice plays for two teams; she outlives any team
print("Red:",  red.players)
print("Blue:", blue.players)


House owns: [Room('kitchen'), Room('bedroom')]
Red: [Player('Alice'), Player('Bob')]
Blue: [Player('Alice')]


## Try it — model a Loan

Extend the Library so a `User` can **borrow** a `Book`. This is an *association* — users
and books exist independently of any loan.

```
 Library ──◆ Book         User ─── Loan ─── Book
                           (1)      (*)      (1)
```

The reference solution below is one possible answer — try writing yours **first**, then compare.


In [5]:
from datetime import date, timedelta

@dataclass
class User:
    name: str

@dataclass
class Loan:
    # association: Loan *references* a User and a Book, it does not own them
    user: User
    book: Book
    due: date
    returned: bool = False

class NotificationSvc:
    def remind(self, user: User, book: Book):
        print(f"  ✉  reminding {user.name} to return {book.title!r}")

class LoanDesk:
    def __init__(self, library: Library, notifier: NotificationSvc):
        self.library = library
        self.notifier = notifier
        self.loans: list[Loan] = []

    def borrow(self, user: User, book: Book) -> Loan:
        if book not in self.library.books:
            raise ValueError(f"{book.title!r} not in library")
        loan = Loan(user, book, due=date.today() + timedelta(days=14))
        self.loans.append(loan)
        self.notifier.remind(user, book)
        print(f"LoanDesk: {user.name} borrowed {book.title!r} (due {loan.due})")
        return loan

desk = LoanDesk(lib, NotificationSvc())
alice = User("Alice")
desk.borrow(alice, lib.books[0])


  ✉  reminding Alice to return 'The Hobbit'
LoanDesk: Alice borrowed 'The Hobbit' (due 2026-05-04)


Loan(user=User(name='Alice'), book=PrintBook('The Hobbit' by J.R.R. Tolkien), due=datetime.date(2026, 5, 4), returned=False)

### Sequence diagram for `desk.borrow(alice, book)`

```
 Caller        LoanDesk        Library         NotificationSvc
   │              │               │                   │
   │──borrow────▶│               │                   │
   │              │── contains? ─▶│                   │
   │              │◀──── yes ─────│                   │
   │              │───────── remind(user,book) ──▶│
   │              │◀────────── ok ─────────────│
   │◀── Loan ─────│               │                   │
```

Notice how each arrow maps to exactly one line of Python in `borrow()`. That's the whole
point of sequence diagrams — they're executable in your head.
